# Combination Methods

Compare 6 combination methods out-of-sample. Does anything beat equal weight?

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.data_bridge import load_all_signal_inputs
from src.signal_diagnostics import compute_ic_timeseries
from src.combination import (
    combine_equal_weight, combine_inverse_ic_vol,
    combine_ols, combine_ridge, combine_elastic_net, combine_bma,
)
from src.walkforward import walkforward_evaluate, compare_methods

## Load panel and outcome

In [ ]:
panel = pd.read_parquet("../data/processed/signal_panel.parquet")
panel["date"] = pd.to_datetime(panel["date"])
sig_cols = [c for c in panel.columns if c not in ("ticker", "date")]

data    = load_all_signal_inputs(n_tickers=100, start="2012-01-01", end="2024-12-31")
prices  = data["prices"]
wide    = prices.pivot(index="date", columns="ticker", values="adj_close")
qprices = wide.resample("QE").last()
fwd     = qprices.shift(-1) / qprices - 1
outcome_df = fwd.stack().reset_index()
outcome_df.columns = ["date", "ticker", "outcome"]
outcome_df = outcome_df.dropna()

quarter_ends = set(outcome_df["date"].unique())
panel_q = (
    panel[panel["date"].isin(quarter_ends)]
    .merge(outcome_df, on=["ticker", "date"], how="inner")
)

print(f"Panel: {panel_q.shape}  |  signals: {sig_cols}")

## Run compare_methods() — 5 standard methods

In [ ]:
# equal_weight doesn't accept outcome_col — wrap it
def ew_fn(panel, signal_cols, outcome_col="outcome"):
    return combine_equal_weight(panel, signal_cols)

methods = {
    "equal_weight": (ew_fn,             {}),
    "ols":          (combine_ols,        {}),
    "ridge":        (combine_ridge,      {"alpha": 1.0}),
    "elastic_net":  (combine_elastic_net, {"alpha": 0.1, "l1_ratio": 0.5}),
    "bma":          (combine_bma,        {}),
}

summary = compare_methods(panel_q, sig_cols, "outcome", methods, min_train_periods=12)
print(summary.to_string(index=False,
    formatters={
        "mean_ic": "{:+.4f}".format,
        "std_ic":  "{:.4f}".format,
        "ir":      "{:+.3f}".format,
        "pct_positive": "{:.0%}".format,
    }
))

## inverse_ic_vol — custom walk-forward loop

This method takes pre-computed IC dicts rather than a panel, so it can't go through `compare_methods`. We compute IC from training data at each step.

In [ ]:
MIN_TRAIN = 12
dates = sorted(panel_q["date"].unique())
iiv_results = []

for i, eval_date in enumerate(dates):
    if i < MIN_TRAIN:
        continue
    train = panel_q[panel_q["date"].isin(dates[:i])]

    # IC timeseries per signal, from training data only
    ic_dict = {}
    for col in sig_cols:
        sig_df = train[["ticker", "date", col]].dropna().rename(columns={col: "signal"})
        out_tr  = train[["ticker", "date", "outcome"]]
        ic_dict[col] = (
            compute_ic_timeseries(sig_df, out_tr)
            if len(sig_df) >= 50 else pd.Series([0.0])
        )

    weights = combine_inverse_ic_vol(ic_dict, sig_cols, window=12)

    eval_clean = panel_q[panel_q["date"] == eval_date].dropna(subset=sig_cols + ["outcome"])
    if len(eval_clean) < 10:
        continue

    X = eval_clean[sig_cols].values.copy()
    for j in range(X.shape[1]):
        X[np.isnan(X[:, j]), j] = np.nanmedian(X[:, j])

    ic, _ = spearmanr(X @ weights, eval_clean["outcome"].values)
    iiv_results.append({"date": eval_date, "composite_ic": ic})

iiv_df = pd.DataFrame(iiv_results)
iiv_ic  = iiv_df["composite_ic"]
iiv_row = pd.DataFrame([{
    "method": "inverse_ic_vol",
    "mean_ic": iiv_ic.mean(),
    "std_ic":  iiv_ic.std(),
    "ir":      iiv_ic.mean() / iiv_ic.std() if iiv_ic.std() > 0 else 0,
    "pct_positive": (iiv_ic > 0).mean(),
    "n_periods": len(iiv_ic),
}])

all_summary = pd.concat([summary, iiv_row], ignore_index=True).sort_values("mean_ic", ascending=False)
print("\n--- Full summary table (all 6 methods) ---")
print(all_summary.to_string(index=False,
    formatters={
        "mean_ic": "{:+.4f}".format,
        "std_ic":  "{:.4f}".format,
        "ir":      "{:+.3f}".format,
        "pct_positive": "{:.0%}".format,
    }
))

## Cumulative IC — all 6 methods on one chart

In [ ]:
# Collect per-period IC series for each method
ic_series = {}
wf_results = {}

for name, (fn, kwargs) in methods.items():
    res = walkforward_evaluate(
        panel_q, sig_cols, "outcome", fn,
        min_train_periods=MIN_TRAIN, **kwargs
    )
    res = res.set_index("date")
    ic_series[name] = res["composite_ic"]
    wf_results[name] = res

ic_series["inverse_ic_vol"] = iiv_df.set_index("date")["composite_ic"]

fig, ax = plt.subplots(figsize=(13, 5))
colors = ["black", "steelblue", "darkorange", "green", "purple", "red"]
styles = ["-", "-", "-", "-", "--", "--"]

for (name, ic), color, ls in zip(ic_series.items(), colors, styles):
    cum = ic.cumsum()
    ax.plot(cum.index, cum.values, label=name, color=color, linestyle=ls, linewidth=1.5)

ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.set_title("Cumulative IC — all 6 combination methods")
ax.set_ylabel("Cumulative Spearman IC")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Ridge weight trajectories

In [ ]:
ridge_weights = pd.DataFrame(
    wf_results["ridge"]["weights"].tolist(),
    index=wf_results["ridge"].index,
)

fig, ax = plt.subplots(figsize=(13, 5))
for col in ridge_weights.columns:
    ax.plot(ridge_weights.index, ridge_weights[col], label=col, linewidth=1.2)
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
ax.set_title("Ridge weights over walk-forward steps")
ax.set_ylabel("Weight")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Does anything beat equal weight out of sample?

On synthetic random data, no method beats equal weight in a meaningful or consistent way — all mean ICs cluster near zero. This is the expected result: equal weight is the maximum-entropy combination and requires no parameter estimation, so it avoids the estimation error that hurts data-driven methods when the signals themselves carry no true predictive content.

**Note on elastic_net:** with `alpha=0.1, l1_ratio=0.5`, the L1 penalty zeroes out all signal weights on random data — the correct solution when nothing predicts the outcome is the zero vector. This produces a constant composite and NaN IC on almost every period (`std=NaN, pct_positive=0%`). On real data with genuinely predictive signals, elastic net would retain a sparse subset rather than collapsing entirely.

The honest finding from real equity data is similar: equal weight is the hardest benchmark to beat, especially out-of-sample. Ridge and elastic net sometimes improve *stability* (lower IC variance → higher IR) without improving mean IC. BMA can exploit genuine signal-level skill differences if they persist out of sample, but on short histories the posterior barely moves from the prior. Inverse-IC-vol benefits most when signal quality is heterogeneous across time, a condition that rarely holds stably. In practice, no combination method consistently beats 1/N by enough to justify the added complexity — which is itself a valid research finding.